# 01 — NL to IR (Natural Language → Intermediate Representation)

This notebook converts a natural-language test case into a structured IR (Intermediate Representation).

Pipeline:
NL → IR → Mapped IR → Code → Execution → Analysis

This notebook:
- Loads a TodoMVC natural-language test case
- Converts it into IR using a minimal rule-based parser
- Saves the IR for downstream notebooks


In [32]:
from pathlib import Path
import json
import re


## 1. Load Natural-Language Test Case

We load a simple test case from the sample-data directory.


In [33]:
sample_path = Path("../sample-data/test_cases/test_home_page.txt")

with open(sample_path, "r") as f:
    test_case_text = f.read()

test_case_text


'Test Name: Home Page Test\nGoal: Verify the home page loads and shows expected content.\nURL: https://ai-test-automation-demo.onrender.com/\n\nSteps:\n1. Navigate to the home page.\n2. Verify the page loads successfully.\n3. Verify the heading "Hello from your Flask Demo App!" is visible.\n4. Verify the "Login" link is visible.\n5. Click the "Login" link.\n6. Verify the browser navigates to /login.\n\nExpected Result:\nThe home page loads and displays the correct heading and login link.\n'

## 2. Minimal NL → IR Converter

This is a lightweight rule-based parser designed to support the TodoMVC demo.

Supported actions:
- Add a todo item
- Press Enter
- Verify the item appears


In [34]:
def generate_ir_from_text(text):
    text_lower = text.lower()
    steps = []

    # Navigation
    if "navigate" in text_lower and "home" in text_lower:
        steps.append({"action": "navigate", "target": "url", "value": "https://ai-test-automation-demo.onrender.com/"})

    if "navigate" in text_lower and "login" in text_lower:
        steps.append({"action": "navigate", "target": "url", "value": "https://ai-test-automation-demo.onrender.com/login"})

    if "navigate" in text_lower and "dashboard" in text_lower:
        steps.append({"action": "navigate", "target": "url", "value": "https://ai-test-automation-demo.onrender.com/dashboard"})

    # Login actions
    if "enter username" in text_lower or "username" in text_lower:
        steps.append({"action": "input", "target": "username_field", "value": "demo"})

    if "enter password" in text_lower or "password" in text_lower:
        steps.append({"action": "input", "target": "password_field", "value": "password123"})

    if "click" in text_lower and "login" in text_lower:
        steps.append({"action": "click", "target": "login_button"})

    # Assertions
    if "welcome to the dashboard" in text_lower:
        steps.append({"action": "assert_text", "target": "dashboard_heading", "value": "Welcome to the Dashboard!"})

    if "invalid username or password" in text_lower:
        steps.append({"action": "assert_text", "target": "error_message", "value": "Invalid username or password"})

    # Navigation links
    if "back to home" in text_lower:
        steps.append({"action": "click", "target": "back_to_home_link"})

    if "logout" in text_lower:
        steps.append({"action": "click", "target": "logout_link"})

    return {
        "test_name": "render_app_test",
        "description": text,
        "steps": steps,
        "metadata": {"priority": "high"}
    }


## 3. Generate IR


In [35]:
ir = generate_ir_from_text(test_case_text)
ir


{'test_name': 'render_app_test',
 'description': 'Test Name: Home Page Test\nGoal: Verify the home page loads and shows expected content.\nURL: https://ai-test-automation-demo.onrender.com/\n\nSteps:\n1. Navigate to the home page.\n2. Verify the page loads successfully.\n3. Verify the heading "Hello from your Flask Demo App!" is visible.\n4. Verify the "Login" link is visible.\n5. Click the "Login" link.\n6. Verify the browser navigates to /login.\n\nExpected Result:\nThe home page loads and displays the correct heading and login link.\n',
 'steps': [{'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/'},
  {'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/login'},
  {'action': 'click', 'target': 'login_button'}],
 'metadata': {'priority': 'high'}}

## 4. Save IR for Downstream Notebooks

The IR is saved to `../sample-data/ir_examples/todomvc_ir.json`.


In [36]:
output_path = Path("../sample-data/ir_examples/home_page_ir.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(ir, f, indent=4)

output_path


PosixPath('../sample-data/ir_examples/home_page_ir.json')